# EDA nhu cầu bán hàng (dữ liệu synthetic)

> ⚠️ Dữ liệu trong notebook này là **synthetic** (sinh từ `data/generate_synthetic.py`).
> Kết quả chỉ dùng để kiểm tra pipeline và hiểu mùa vụ, **không** đại diện cho
> độ chính xác thương mại.

Mục tiêu:
1. Kiểm tra cấu trúc dữ liệu bán hàng theo ngày.
2. Xem mùa vụ theo thứ trong tuần và xu hướng.
3. Đo tỉ lệ ngày không bán, mức độ biến động theo SKU — yếu tố ảnh hưởng WAPE.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "synthetic"
DATA

In [ ]:
sales = pd.read_csv(DATA / "sales.csv", parse_dates=["date"])
products = pd.read_csv(DATA / "products.csv")
inventory = pd.read_csv(DATA / "inventory.csv")

print("sales rows:", len(sales), "| SKUs:", sales.sku.nunique())
print("date range:", sales.date.min().date(), "→", sales.date.max().date())
sales.head()

In [ ]:
daily = sales.groupby("date", as_index=False).agg(units=("quantity", "sum"), revenue=("quantity", lambda s: 0))
daily["revenue"] = sales.assign(revenue=sales.quantity * sales.unit_price).groupby("date")["revenue"].sum().values
print("Ngày có bán:", len(daily), "| trung bình units/ngày:", round(daily.units.mean(), 2))
daily.tail()

## 1. Xu hướng nhu cầu theo ngày

Đường trung bình động 7 ngày giúp tách nhiễu cuối tuần khỏi xu hướng dài hạn.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily.date, daily.units, color="#9aa5b1", linewidth=1, label="Units/ngày")
ax.plot(daily.date, daily.units.rolling(7).mean(), color="#2563eb", linewidth=2, label="MA 7 ngày")
ax.set_title("Tổng nhu cầu theo ngày (synthetic)")
ax.set_ylabel("Số lượng")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Mùa vụ theo thứ trong tuần

Nếu cuối tuần cao hơn rõ rệt, mô hình cần feature thứ/ngày trong tuần — engine hiện dùng
`dayofweek`, `is_weekend` và baseline seasonal naive theo thứ.

In [ ]:
sales["weekday"] = sales.date.dt.day_name()
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
by_weekday = sales.groupby("weekday").quantity.mean().reindex(order)

ax = by_weekday.plot(kind="bar", figsize=(9, 3.5), color="#2563eb")
ax.set_title("Số lượng bán trung bình theo thứ")
ax.set_ylabel("Units / dòng bán")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
by_weekday.round(2)

## 3. Đặc điểm theo SKU

- `zero_share`: tỉ lệ ngày không phát sinh bán — cao làm dự báo khó và WAPE tăng.
- `cv`: hệ số biến thiên (std/mean) — đo độ nhiễu.
- Các SKU bán chậm thường được baseline chọn thay vì ML.

In [ ]:
full_range = pd.date_range(sales.date.min(), sales.date.max(), freq="D")
rows = []
for sku, group in sales.groupby("sku"):
    series = group.set_index("date").quantity.reindex(full_range, fill_value=0)
    rows.append(
        {
            "sku": sku,
            "days_with_sales": int((series > 0).sum()),
            "zero_share": round(float((series == 0).mean()), 3),
            "avg_daily": round(float(series.mean()), 2),
            "std_daily": round(float(series.std(ddof=0)), 2),
            "cv": round(float(series.std(ddof=0) / series.mean()), 2) if series.mean() else None,
        }
    )

stats = pd.DataFrame(rows).merge(products[["sku", "name", "category"]], on="sku")
stats.sort_values("avg_daily", ascending=False).head(10)

In [ ]:
ax = stats.plot.scatter(x="avg_daily", y="zero_share", figsize=(8, 4), color="#f97316")
ax.set_title("Nhu cầu trung bình vs tỉ lệ ngày không bán")
ax.set_xlabel("Units/ngày (trung bình)")
ax.set_ylabel("Tỉ lệ ngày không bán")
plt.tight_layout()
plt.show()

stats.sort_values("zero_share", ascending=False).head(8)[["sku", "name", "avg_daily", "zero_share"]]

## 4. Tồn kho & rủi ro

So sánh `on_hand` với nhu cầu bình quân để thấy các SKU mỏng tồn kho — đây là đầu vào
cho decision engine (ROP/safety stock) trong backend.

In [ ]:
risk = inventory.merge(stats[["sku", "avg_daily", "name"]], on="sku")
risk["days_of_cover"] = (risk.on_hand / risk.avg_daily.replace(0, pd.NA)).round(1)
risk.sort_values("days_of_cover").head(10)[["sku", "name", "on_hand", "avg_daily", "days_of_cover"]]

## 5. Ghi chú cho mô hình

- Tỉ lệ ngày không bán cao giải thích vì sao WAPE trên dữ liệu synthetic khá lớn (~0,55).
- Nên đánh giá riêng nhóm SKU bán chạy và nhóm bán chậm.
- Khi có dữ liệu thật: thêm giá/khuyến mãi/ngày lễ chính xác, và theo dõi drift theo tháng.
- Chạy `POST /api/forecasts/run` để xem mô hình nào được chọn cho từng SKU.
